# Fine Tuning VS Full Training a model

Let's take a model like BERT, which has been pretrained for masked language modeling (predict masked words) and next sequence prediction (predict if two sentences were following each other or not), and prepare it for a different task: to determine whether two sentences are paraphrases of each other.

We will first fine-tune it and later completely re-train it again, so we understand how both processes work and what are the outcomes.

## Install libraries

In [ ]:
!pip install -q datasets transformers evaluate fsspec gcsfs --upgrade

## Import libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from pprint import pprint

## Download datasets

Download the Microsoft Research Paraphrase Corpus (MRPC) task from the General Language Understanding Evaluation (GLUE) benchmark, so we can use it for a paraphrase detection task where the goal is to determine whether two sentences in a pair are semantically equivalent. It includes datasets for _train_, _validation_ and _test_.

In [ ]:
raw_datasets = load_dataset("glue", "mrpc", download_mode="force_redownload")
print(raw_datasets)
pprint(raw_datasets["train"].features)
pprint(raw_datasets["train"][0])

## Pre-process data

Tokenize the datasets sentences to convert the text in all 3 datasets to numbers. `tokenizer` returns a dictionary that needs to be stored in RAM, while the downloaded datasets are stored in a folder in your HD. The `.map` method scales well for very large datasets, as it can process data in chunks without requiring the entire dataset to be held in memory. It is better suited for large, streamed datasets or when leveraging dataset libraries, providing scalability and memory efficiency.

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)   # the model needs pairs of sentences --- and look... no padding!

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
print(tokenized_datasets)     # the function ADDED 3 new fields to the dataset: 'input_ids', 'token_type_ids', 'attention_mask'


We’ve left the `padding` argument out because padding all the samples to the maximum length is not efficient: let's pad to the maximum length in a batch, and not the maximum length in the entire dataset. The collate function does exactly that and we delay it to the very end of the data pre-processing phase.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Let's see how it works on the first 8 entries.

In [ ]:
samples = tokenized_datasets["train"][:8]   # select the first 8 entries
print(samples)

samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}    # remove the text columns and the index
print([len(x) for x in samples["input_ids"]])

batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

## Fine Tuning

The only required parameter is the name of the directory where the trained model will be saved.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer", report_to="none")   # no logging so no need for wandb

Now we load the BERT model.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

You receive a warning because BERT is pre-trained for tasks like masked language modeling (predicting masked words) and next sentence prediction (determining if two sentences follow each other). However, it has not been pre-trained specifically for detecting paraphrasing pairs of sentences. While BERT has already learned general language patterns from a large dataset, additional training is required to fine-tune it for paraphrasing analysis. This is exactly what we intend to accomplish with the fine-tuning.

The `Trainer` class is used to fine-tune this pre-trained model on your specific dataset. Fine-tuning involves slightly adjusting the model's pre-learned weights to better fit the new task, like sentiment analysis or named entity recognition, using your dataset. Let's instantiate it using everything we have built up to now: the model, the training_args, the training and validation datasets, data_collator, and tokenizer.

The `Trainer` class abstracts much of the boilerplate code needed for setting up training loops, handling data, managing devices, and evaluating the model. It provides built-in features for logging, saving, and evaluation, making fine-tuning accessible and efficient.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

Let's evaluate the model before fine-tuning it.

In [ ]:
metrics_before_finetuning = trainer.evaluate()
print("Evaluation metrics before fine-tuning:")
print(metrics_before_finetuning)

As you can see, by default it just reports on loss, so there is no guidance on more meaningful data (e.g. accuracy or f1). To get that information we need to provide an additional parameter to our `Trainer` instance: `compute_metrics`, a function we will build using the `evaluate` library to return the required metrics.

We start by loading the same MRPC dataset into `evaluate` so it knows what are the correct labels the predictions need to be compared with.

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")

`evaluate` needs some predictions to work with, so we will use the `trainer.predict()` method with the "validation" dataset to generate them.

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

Now we have 408 entries with 2 logits each (raw predictions), and 1 true label each (0 for not paraphrase, 1 for paraphrase).

In [ ]:
print(predictions.predictions[0])
print(predictions.label_ids[0])

As long as the returned predictions are 2 logits, one for each class (first one signalling the probability of *not* being a paraphrase, and the second one the probability of being a paraphrase), we need to identify which one (what column, or last axis) has a higher value or probability.

In [ ]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)
print(preds)

Let's compare those predictions against the labels to determine how accurate the model is before fine-tuning it.

In [ ]:
metric.compute(predictions=preds, references=predictions.label_ids)

Pretty bad predictions, huh? Our model definitely needs some fine-tuning to get better at the paraphrasing task we want it to accomplish.

Now we have everything we need to create the required `compute_metrics()` function we mentioned.

In [ ]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

We can now use that function as a parameter to instantiate the `Trainer` class...

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

... and check again the model accuracy before fine-tuning, but this time with meaningful information and data points.

In [ ]:
metrics_before = trainer.evaluate()

print("\n")
print("Accuracy = How often was the model right about the paraphrases?")
print("   (TP + TN) / (TP + FP + FN + TN)")
print("F1 = How well did the model detect paraphrases without missing or falsely labeling them?")
print("\n")
print("F1 is a combination of precision and recall:")
print("Precision = Of all detected paraphrases, how many were right? (correctness of POSITIVE guesses)")
print("   TP / (TP + FP)")
print("Recall = Of all the paraphrases in the dataset, how many did the model detect? (coverage of actual positives)")
print("   TP / (TP + FN)")
print("\n")

print("Evaluation metrics BEFORE fine-tuning:")
# print(metrics_before)
print("Loss:", metrics_before["eval_loss"])
print("Accuracy:", metrics_before["eval_accuracy"])
print("F1:", metrics_before["eval_f1"])

We can now fine-tune our model... (this might take 5 mins)

In [ ]:
trainer.train()

 ... with insights about its accuracy.

In [ ]:
print("Evaluation metrics BEFORE fine-tuning:")
print("Loss:", metrics_before["eval_loss"])
print("Accuracy:", metrics_before["eval_accuracy"])
print("F1:", metrics_before["eval_f1"])
print("\n")

metrics_after = trainer.evaluate()

print("\n")
print("Evaluation metrics AFTER fine-tuning:")
print("Loss:", metrics_after["eval_loss"])
print("Accuracy:", metrics_after["eval_accuracy"])
print("F1:", metrics_after["eval_f1"])

## Full Training

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

Load the training and validation subsets.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

Check that padding is maxed for each batch.

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

When instantiating the model you get a warning because BERT has been pretrained for masked language modeling (predict masked words) and next sequence prediction (predict if two sentences were following each other or not), but it has NOT been pretrained on classifying pairs of sentences. It has already been pre-trained on a large dataset, so it has already learned general language patterns and structures. So we need to TRAIN it for sentiment analysis.

Let's check how the model does with the first batch before being trained.

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

Overall loss seems to be pretty high, so it clearly needs to be trained before it can perform well in paraphrasing detection.

There are several things we need to define before training the model, starting with the optimizer we will use. The optimizer is an algorithm used to adjust the parameters of a model (such as weights and biases) to minimize a loss function during the training process. The optimizer is essential in training deep learning models, as it implements the iterative process of gradient descent to update parameters, minimize the loss function, and enable the model to learn from data. For our demos we will use an optimizer called `AdamW`.

`lr` defines the learning rate.

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

 Then we need to define a learning rate scheduler, an important tool in the optimization process, as they adjust the learning rate during training based on a predefined schedule or policy. This can help improve convergence and performance. `get_scheduler` is used to obtain a learning rate scheduler to train our model.


In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

If we want to perform our model training in a reasonable time we will need to make sure we use an available GPU.

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

With all these parameters correctly configure we are finally ready to train our model.

This is a custom training loop designed to train a model from scratch. The loop manually handles the forward and backward passes, gradient updates, and learning rate adjustments. While you are using an already pre-trained model like BERT, the detailed manual control implies that you are performing significant retraining (almost like training it from scratch). This means heavily modifying its pre-trained architecture, so it requires more computational resources and time, as the model needs to learn both general and task-specific patterns entirely from the data provided.


In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()                                               # switch the model to training mode so udpates can occur
for epoch in range(num_epochs):                             # each epoch is a full pass through the entire dataset
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()} # move keys and values (tensors) to GPU
        outputs = model(**batch)                            # forward pass: unpack the dict and have the model calculate predictions
        loss = outputs.loss                                 # compute loss: measure how well the model's predictions match the actual labels (lower is better)
        loss.backward()                                     # backward pass (backpropagation): compute the gradients of the loss with respect to the model parameters
                                                            # gradients indicate how much each parameter should be adjusted to minimize the loss
        optimizer.step()                                    # update the model parameters using the calculated gradients to reduce the loss
        lr_scheduler.step()                                 # adjust learning rate to improve convergence
        optimizer.zero_grad()                               # reset gradients
        progress_bar.update(1)

We can now evaluate how good the model predictions are, using the Microsoft Research Paraphrase Corpus (MRPC) task from the General Language Understanding Evaluation (GLUE) benchmark. They are used for a paraphrase detection task where the goal is to determine whether two sentences in a pair are semantically equivalent.

We will use the labels in the evaluation data subset to check how well the model did in predicting paraphrases.

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()                                              # set the model to evaluation mode
for batch in eval_dataloader:                             # for every batch of 8 in the evaluation subset
    batch = {k: v.to(device) for k, v in batch.items()}   # move them to GPU
    with torch.no_grad():                                 # disable gradient tracking (not needed during evaluation)
        outputs = model(**batch)                          # forward pass: calculate predictions

    logits = outputs.logits                               # raw unnormalized preditions
    predictions = torch.argmax(logits, dim=-1)            # determine what column has the highest value
    metric.add_batch(predictions=predictions, references=batch["labels"])     # accumulate predictions and labels for computation of the overall metric

metric.compute()    # final evaluation metric using the accumulated predictions and references (accuracy and F1 score)

**Not bad!**